In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
import numpy as np
import pandas as pd
import torch
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
df = pd.read_csv(
    "/content/drive/MyDrive/Customer_project/final_02_comp.csv"
)

df.head()

In [ ]:
label_encoder = joblib.load(
    "/content/drive/MyDrive/Customer_project/models/label_encoder.pkl"
)

In [ ]:
df = df.sample(
    n=30000,
    random_state=42
).reset_index(drop=True)

print(df.shape)

In [ ]:
import joblib

label_encoder = joblib.load(
    "/content/drive/MyDrive/Customer_project/models/label_encoder.pkl"
)

In [ ]:
X = df["Processed_Complaint"]
y = label_encoder.transform(df["Product"])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

In [ ]:
train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

In [ ]:
train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": y_train.tolist()
})

test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
    "labels": y_test.tolist()
})

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_encoder.classes_)
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
training_args = TrainingArguments(

    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    logging_dir="./logs",

    logging_steps=100,

    report_to="none"

)

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics

)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()

print(results)

In [ ]:
trainer.save_model(
    "/content/drive/MyDrive/Customer_project/models/distilbert_model"
)

tokenizer.save_pretrained(
    "/content/drive/MyDrive/Customer_project/models/distilbert_model"
)

print("DistilBERT Model Saved Successfully!")

In [ ]:
import pandas as pd

results_df = pd.DataFrame([{
    "Model": "DistilBERT",
    "Accuracy": results["eval_accuracy"],
    "Precision": results["eval_precision"],
    "Recall": results["eval_recall"],
    "F1 Score": results["eval_f1"]
}])

results_df.to_csv(
    "/content/drive/MyDrive/Customer_project/distilbert_results.csv",
    index=False
)

results_df

In [ ]:
print(model)